# Olist — preparação e engenharia de features

Ciência dos Dados (T326), UNIFOR. Este notebook consulta os artefatos do pipeline, sem EDA completa ou modelagem. Execute `python -m olist_features.pipeline` no ambiente local antes de abrir. O relatório técnico está em `docs/engenharia_features.md`.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/olist_features").is_dir())
assert Path(sys.prefix).resolve() == (ROOT / ".venv").resolve(), "Selecione Python (Olist .venv)"
REPORTS = ROOT / "resultados/tabelas"
manifest = json.loads((REPORTS / "run_manifest.json").read_text(encoding="utf-8"))
print("Python:", sys.version.split()[0])
print("Pedidos antes/depois:", manifest["rows_before"], manifest["rows_after"])
print("Modelos treinados:", manifest["trained_models"])

Python: 3.13.6
Pedidos antes/depois: 99441 96470
Modelos treinados: 0


## Coorte rotulada

Unidade: pedido. Alvo: segundos entre compra e recebimento divididos por 86.400. Status e recebimento são usados apenas para seleção e rótulo. A cobertura abaixo tem exclusões sequenciais, sem dupla contagem.

In [2]:
display(pd.read_csv(REPORTS / "coverage.csv"))

,stage,remaining,excluded_at_stage
0,raw_orders,99441,0
1,status_not_delivered,96478,2963
2,invalid_purchase_timestamp,96478,0
3,invalid_delivery_timestamp,96470,8
4,negative_duration,96470,0
5,no_items,96470,0


## Contrato de entradas

As features usam compra, carrinho e cadastros sob hipótese de vigência histórica. Pagamentos e avaliações são somente auditados. Consulte o dicionário para fórmulas e justificativas.

In [3]:
contract = json.loads((ROOT / "dados/tratados/feature_contract.json").read_text(encoding="utf-8"))
data = pd.read_parquet(ROOT / "dados/tratados/olist_orders_features.parquet")
feature_columns = contract["feature_columns"]
assert data.order_id.is_unique
assert contract["target_column"] not in feature_columns
assert list(data.columns) == [contract["id_column"], *feature_columns, contract["target_column"]]
print(f"{len(data)} linhas; {len(feature_columns)} features; uma linha por pedido.")
display(pd.read_csv(REPORTS / "feature_dictionary.csv"))

96470 linhas; 30 features; uma linha por pedido.


,feature,source,description,dtype,availability
0,products_value_total,items.price,Soma dos preços dos itens; BRL.,float64,compra/cadastro; hipótese de vigência histórica
1,freight_value_total,items.freight_value,Soma do frete dos itens; BRL.,float64,compra/cadastro; hipótese de vigência histórica
2,order_value_total,"items.price, items.freight_value",Produtos + frete; não usa pagamentos.,float64,compra/cadastro; hipótese de vigência histórica
3,items_count,items.order_item_id,"Número de linhas de itens, não máximo do seque...",float64,compra/cadastro; hipótese de vigência histórica
4,products_count,items.product_id,Quantidade de produtos distintos no carrinho.,float64,compra/cadastro; hipótese de vigência histórica
5,sellers_count,items.seller_id,Quantidade de vendedores distintos.,float64,compra/cadastro; hipótese de vigência histórica
6,seller_states_count,sellers.seller_state,Quantidade de UFs dos vendedores.,float64,compra/cadastro; hipótese de vigência histórica
7,same_state_items_fraction,"customers.customer_state, sellers.seller_state",Fração de itens com vendedor na UF do comprador.,Float64,compra/cadastro; hipótese de vigência histórica
8,weight_g_total,products.product_weight_g,Soma por item; ausente se houver peso ausente/...,float64,compra/cadastro; hipótese de vigência histórica
9,weight_g_mean,products.product_weight_g,Média dos pesos conhecidos por item.,float64,compra/cadastro; hipótese de vigência histórica


## Qualidade e verificação

Ausências e extremos foram preservados e sinalizados. Não há imputação ou transformação aprendida nesta etapa. As auditorias contêm informações posteriores e ficam fora de X.

In [4]:
missing = pd.read_csv(REPORTS / "missing_final.csv")
display(missing.loc[missing.missing.gt(0)])
checks = pd.read_csv(REPORTS / "verification_checks.csv")
assert checks.passed.all()
display(checks)
print("Integridade original preservada:", manifest["input_integrity_before"] == manifest["input_integrity_after"])

,column,missing,missing_pct,distinct_non_null
9,weight_g_total,22,0.022805,2858
10,weight_g_mean,22,0.022805,2766
11,volume_cm3_total,16,0.016585,6262
12,length_cm_max,16,0.016585,99
13,height_cm_max,16,0.016585,102
14,width_cm_max,16,0.016585,94
20,main_category,1332,1.380740,73
23,distance_km_mean,477,0.494454,90374
24,distance_km_min,477,0.494454,90283
25,distance_km_max,477,0.494454,90268


,check,passed,evidence
0,unique_order_id,True,96470
1,features_exact_allowlist,True,30
2,dataset_exact_allowlist,True,32
3,finite_nonnegative_target,True,dias fracionários
4,items_count_independent,True,groupby.size
5,products_value_total_independent,True,soma independente antes das junções
6,freight_value_total_independent,True,soma independente antes das junções
7,order_value_identity,True,produtos + frete
8,target_datetime_independent,True,96470
9,label_alignment,True,merge one_to_one e ordenação


Integridade original preservada: True


## Próximo acompanhamento

Validar coorte de entregues, vigência dos cadastros, aproximação por CEP e política futura para ausências/extremos. Divisão temporal, EDA, treinamento e otimização ficam para outra tarefa.